In [1]:
import requests
import pandas as pd
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import mexc_spot_v3

hosts = "https://api.mexc.com"
mexc_key = ""
mexc_secret = ""
market = mexc_spot_v3.mexc_market(mexc_hosts=hosts)

command_executor = 'http://185.215.187.158:4444'
options = webdriver.ChromeOptions()
options.add_argument('--ignore-ssl-errors=yes')
options.add_argument('--ignore-certificate-errors')


In [3]:
def new_tickers():
    tickers = market.get_24hr_ticker()
    return [ticker['symbol'] for ticker in tickers if ticker['lastPrice'] == '0']

new_tickers()

[]

In [4]:
def tickers_in_api():
    tickers = new_tickers()
    default_tickers = market.get_defaultSymbols()['data']
    return {ticker:True if ticker in default_tickers else False for ticker in tickers}

tickers_in_api()

{}

In [10]:
def server_time():
    server_time = market.get_timestamp()
    server_time = server_time['serverTime']/1000
    return datetime.fromtimestamp(server_time)

In [23]:
diff_times = []
for i in range(1000):
    s_time = server_time()
    now = datetime.now()
    diff_times.append((now - s_time).total_seconds()*1000000)
    time.sleep(0.1)

mean_time = sum(diff_times)/len(diff_times)
max_time = max(diff_times)
min_time = min(diff_times)
print(min_time, mean_time, max_time)

118888.0 141912.333 463184.0


In [27]:
with open('diff_times.txt','w') as file:
    file.write(str(diff_times))

In [ ]:
def launch_times():
    driver = webdriver.Remote(
    command_executor=command_executor,
    options=options
    )
    launch_time = {}
    driver.maximize_window()
    try:
        new_tickers = new_tickers()
        for ticker in new_tickers:
            url = f"https://www.mexc.com/exchange/{ticker.replace('USDT','_USDT')}"
            driver.get(url)
            element = driver.find_element(by=By.CLASS_NAME, value="countDown_deadline__Inua0").text
            launch_time[ticker] = element
        driver.close()
        driver.quit()
    except Exception as e:
        print(e)
        driver.close()
        driver.quit()
    return launch_time

launch_times()